# MNIST — Handwritten Digit Recognition

**Medium** &nbsp;·&nbsp; `Classification` `Neural Networks`

Classify 28×28 greyscale images of handwritten digits into the 10 classes `0`–`9`.

MNIST is 70,000 images: **60,000 train / 10,000 test**, already split for you by
the people who built it. Each image is 784 pixels with values `0`–`255`, and each
carries one integer label.

The task looks small and it is — that is exactly why it is the standard first
problem. It is big enough that a bad idea visibly loses and small enough that you
can build four different models in one notebook and watch the accuracy climb.

---

**Requirements:**

- classify all 10,000 test images
- **NumPy only** — no sklearn, no torch, no scipy. You write the backprop.
- report accuracy on the **test** set, which no model is allowed to train on

---

### The four models, in order

Build them in this order. Each one exists to fix a specific thing the previous
one got wrong, and the jump between them is the actual lesson.

| | Model | Idea | Target |
|---|---|---|---|
| **V1** | Nearest centroid | average each digit into one template, pick the closest | ~0.82 |
| **V2** | k-NN | remember every training image, vote among the `k` closest | ~0.94 |
| **V3** | Softmax regression | one linear layer, trained by gradient descent | ~0.92 |
| **V4** | 2-layer neural net | one hidden layer, ReLU, backprop | **>0.97** |

Those targets are what this setup gets, not a ceiling — treat them as "am I in
the right neighbourhood". V3 scoring *below* V2 is expected, not a bug; the
closing section is about why that trade is still worth making.

The data loader below is plumbing, not the exercise — run it and move on.

## Loading the data — given, run it and don't edit it

The IDX format is a 4-byte magic number, then the dimension sizes as big-endian
`int32`, then the raw bytes. `np.frombuffer` reads the whole thing in one go.

First run downloads ~11 MB and caches to `../data/mnist.npz`; every run after
that reads the cache.

In [ ]:
import gzip
import os
import struct
import urllib.request

import numpy as np

DATA_DIR = "../data"
CACHE = os.path.join(DATA_DIR, "mnist.npz")

FILES = {
    "train_x": "train-images-idx3-ubyte.gz",
    "train_y": "train-labels-idx1-ubyte.gz",
    "test_x":  "t10k-images-idx3-ubyte.gz",
    "test_y":  "t10k-labels-idx1-ubyte.gz",
}
MIRRORS = [
    "https://ossci-datasets.s3.amazonaws.com/mnist/",
    "https://storage.googleapis.com/cvdf-datasets/mnist/",
]


def parse_idx(raw: bytes) -> np.ndarray:
    """IDX: magic(4) | dim sizes as big-endian int32 | payload bytes."""
    magic, = struct.unpack(">I", raw[:4])
    n_dims = magic & 0xFF                       # low byte of the magic = rank
    dims = struct.unpack(f">{n_dims}I", raw[4:4 + 4 * n_dims])
    return np.frombuffer(raw[4 + 4 * n_dims:], dtype=np.uint8).reshape(dims)


def load_mnist():
    if os.path.exists(CACHE):
        with np.load(CACHE) as d:
            return d["train_x"], d["train_y"], d["test_x"], d["test_y"]

    os.makedirs(DATA_DIR, exist_ok=True)
    arrays = {}
    for key, fname in FILES.items():
        for base in MIRRORS:
            try:
                with urllib.request.urlopen(base + fname, timeout=60) as r:
                    arrays[key] = parse_idx(gzip.decompress(r.read()))
                break
            except Exception as e:                # try the next mirror
                last = e
        else:
            raise RuntimeError(f"could not download {fname}: {last}")
        print(f"downloaded {fname:<30} {arrays[key].shape}")

    np.savez_compressed(CACHE, **arrays)
    return arrays["train_x"], arrays["train_y"], arrays["test_x"], arrays["test_y"]


train_x, train_y, test_x, test_y = load_mnist()

print("train images", train_x.shape, train_x.dtype)
print("train labels", train_y.shape, "values", np.unique(train_y))
print("test  images", test_x.shape)
print("pixel range ", train_x.min(), "-", train_x.max())

## Look at the data before modelling it

Two things worth checking on any dataset before writing a single model: what a
sample actually looks like, and whether the classes are balanced.

Run it, then answer for yourself: is this balanced enough that plain accuracy is
an honest metric, or do you need something else?

In [ ]:
GLYPHS = " .:-=+*#%@"          # 10 ink levels, light to dark

def show(img, label=None):
    for row in img:
        print("".join(GLYPHS[min(int(p) * len(GLYPHS) // 256, len(GLYPHS) - 1)] for p in row))
    if label is not None:
        print(f"label = {label}")

show(train_x[0], train_y[0])

print()
print("class balance (train):")
counts = np.bincount(train_y, minlength=10)
for d, c in enumerate(counts):
    print(f"  {d}  {c:>5}  {'#' * (c * 40 // counts.max())}")

## Preprocessing — your turn

Two transformations, and you should be able to say why each one is needed:

- **flatten** each 28×28 image to a 784-vector. None of these four models knows
  the pixels sit on a grid — that is exactly what a CNN adds later.
- **scale** the pixels from `0–255` into `[0, 1]`, as `float32`. Think about what
  would happen to a gradient-descent step size if you skipped this.

Fill in `Xtr`, `Xte`, `ytr`, `yte` below. Everything after this cell uses them.

In [ ]:
# TODO: flatten to (n, 784) and scale to [0, 1] as float32
Xtr = None
Xte = None
ytr = train_y.astype(np.int64)
yte = test_y.astype(np.int64)

print("Xtr", None if Xtr is None else (Xtr.shape, Xtr.dtype, f"{Xtr.min():.1f}-{Xtr.max():.1f}"))
print("Xte", None if Xte is None else Xte.shape)
print("expected: (60000, 784) float32 0.0-1.0  and  (10000, 784)")

results = {}        # model name -> test accuracy, fill in as you go

## V1 — Nearest centroid

The simplest thing that could possibly work: average all the `0`s into one
template image, all the `1`s into another, and classify a new image by whichever
template it is closest to in Euclidean distance.

Ten 784-dim vectors is the entire model.

**Build it:**

1. `centroids` — shape `(10, 784)`, row `d` is the mean of every training image
   with label `d`.
2. `pred_v1` — for each test image, the index of the nearest centroid.

**On step 2:** the obvious way is a loop over 10,000 test images. Don't. Expand
the squared distance:

$$\|x - c\|^2 = \|x\|^2 - 2\,x \cdot c + \|c\|^2$$

Now look at each of the three terms and ask which ones actually vary as you
change `c` for a fixed test image `x`. One of them does not — so it cannot change
which class wins, and you can drop it entirely. What is left is a single matrix
multiply for the whole test set.

Careful with the sign: dropping a term flips you between `argmin` of a distance
and `argmax` of a score.

In [ ]:
import time
t0 = time.perf_counter()

centroids = None    # TODO: (10, 784), one mean image per digit
pred_v1 = None      # TODO: (10000,) predicted label per test image

if pred_v1 is not None:
    acc = (pred_v1 == yte).mean()
    results["V1 nearest centroid"] = acc
    print(f"V1 nearest centroid  test accuracy {acc:.4f}   ({time.perf_counter() - t0:.1f}s)")
    print("\nwhat the model actually IS - the centroid for 3:")
    show((centroids[3] * 255).reshape(28, 28))

Look at that centroid image once it prints. It is a **blur** — every way of
writing a `3` averaged into one shape. That is the failure mode, and it tells you
what the next model has to fix: a `3` with a flat top looks no more like that
template than a `5` does.

## V2 — k-nearest neighbours

The opposite trade: keep **every** training image and let a new image vote among
its `k` closest neighbours. There is no training at all; all the cost moves to
prediction time.

**Build `knn_predict(Xq, Xref, yref, k)`:**

- Same distance expansion as V1, so the distances are one matmul. Here it is
  `Xq @ Xref.T`, which for the full problem would be a 10,000 × 60,000 matrix —
  hence the `chunk` parameter, so you process a few hundred queries at a time and
  the intermediate never blows up your memory.
- You need the `k` smallest per row. `np.sort` is `O(n log n)` and you are
  throwing away almost all of it. Look up `np.argpartition` — it gets the `k`
  smallest in `O(n)` and does not bother ordering them, which you do not need.
- Then the majority label among those `k`. `np.bincount(...).argmax()` per row.

`N_REF` is the honest knob: using all 60k would score about a point higher and
take ~3× longer. Say what you used.

**Then answer:** the sweep at the bottom tries `k = 1, 3, 5, 9`. Before you run
it, predict whether bigger `k` helps or hurts here, and why.

In [ ]:
def knn_predict(Xq, Xref, yref, k=3, chunk=500):
    """Predict labels for Xq by majority vote among the k nearest rows of Xref."""
    # TODO
    pass


N_REF, N_QUERY = 20_000, 2_000
rng = np.random.default_rng(0)
ref = rng.choice(len(Xtr), N_REF, replace=False) if Xtr is not None else None

t0 = time.perf_counter()
pred_v2 = knn_predict(Xte[:N_QUERY], Xtr[ref], ytr[ref], k=3) if ref is not None else None

if pred_v2 is not None:
    acc = (pred_v2 == yte[:N_QUERY]).mean()
    results[f"V2 k-NN (k=3, {N_REF//1000}k ref)"] = acc
    print(f"V2 k-NN              test accuracy {acc:.4f}   "
          f"({time.perf_counter() - t0:.1f}s for {N_QUERY} queries)")

    print("\ndoes k matter?")
    for k in [1, 3, 5, 9]:
        p = knn_predict(Xte[:1000], Xtr[ref], ytr[ref], k=k)
        print(f"    k={k:<2} {(p == yte[:1000]).mean():.4f}")

A big jump over the centroids, from an algorithm that does no learning
whatsoever. It works because it never averages.

But notice what you just built: there is no model to ship. The "model" *is* 63 MB
of reference images, every query touches all of it, and that cost grows with your
dataset forever. That is the entire argument for the next two — spend time once,
up front, to learn a small fixed set of numbers.

## V3 — Softmax regression

The first model that actually *learns*. One linear layer, `784 → 10`, turned into
probabilities by softmax, fitted by minimising cross-entropy with mini-batch
gradient descent.

$$p = \text{softmax}(xW + b), \qquad \mathcal{L} = -\log p_{y}$$

**Two helpers first:**

- `softmax(z)` for a batch — rows are independent. There is one numerical trap:
  `exp` of a large logit overflows to `inf`. The standard fix is to subtract the
  row max from every row before exponentiating. Convince yourself that this
  cannot change the result (what happens to the shared factor in the numerator
  and denominator?).
- `one_hot(y)` — labels to a `(n, 10)` matrix of 0s and one 1.

**Then the training loop.** Shuffle each epoch, walk the data in mini-batches,
and for each batch: forward, gradient, update.

The gradient is the part worth deriving rather than looking up. For softmax
followed by cross-entropy, the derivative of the loss with respect to the
**logits** collapses to something remarkably clean — `p - onehot(y)`, averaged
over the batch. It is the single tidiest result in this area and it is the reason
those two functions are always paired. Derive it, or at least verify it
numerically before trusting it.

From there `dW = x.T @ dlogits` and `db = dlogits.sum(axis=0)`.

Start with `lr = 0.5`, batch 128, 30 epochs. `W` can be initialised to zeros here
— worth asking yourself why that is fine now but will *not* be in V4.

In [ ]:
def softmax(z):
    """Row-wise softmax, overflow-safe."""
    # TODO
    pass


def one_hot(y, n=10):
    """(n_samples,) int labels -> (n_samples, n) 0/1 matrix."""
    # TODO
    pass


rng = np.random.default_rng(0)
W = np.zeros((784, 10), dtype=np.float32)
b = np.zeros(10, dtype=np.float32)

EPOCHS, BATCH, LR = 30, 128, 0.5
t0 = time.perf_counter()

# TODO: the training loop
#   for each epoch:
#       shuffle the training indices
#       for each mini-batch:
#           forward  -> probabilities
#           gradient -> (p - onehot(y)) / batch_size
#           update   -> W, b

pred_v3 = None      # TODO: argmax of the logits on Xte

if pred_v3 is not None:
    acc = (pred_v3 == yte).mean()
    results["V3 softmax regression"] = acc
    print(f"V3 softmax           test accuracy {acc:.4f}   "
          f"({time.perf_counter() - t0:.1f}s, {W.size + b.size:,} parameters)")

It probably lost to k-NN — and it fits in 31 KB and predicts in one matrix
multiply. Accuracy is not the only axis.

It is stuck where it is for a structural reason worth being able to state: a
linear model can only draw **flat** boundaries between classes, so it is really
still learning one template per digit, just a better-weighted one. `4` vs `9` and
`3` vs `5` are not linearly separable in raw pixel space, and more training will
not fix that.

## V4 — Two-layer neural network

Add one hidden layer with a ReLU and the boundaries can bend.

```
784 -> 256 (ReLU) -> 10 (softmax)
```

**Forward:** `z1 = x@W1 + b1`, `a1 = relu(z1)`, `z2 = a1@W2 + b2`.

**Backward** — chain rule, working right to left. You already know `dz2` from V3
(same softmax + cross-entropy ending). From there:

- `dW2 = a1.T @ dz2`, `db2 = dz2.sum(0)`
- `da1 = dz2 @ W2.T` — push the error back through the second layer
- `dz1 = da1 * (relu derivative at z1)` — the derivative of `max(0, z)` is `1`
  where `z > 0` and `0` elsewhere, so this is a mask
- `dW1 = x.T @ dz1`, `db1 = dz1.sum(0)`

**Three details that are doing real work.** Get these wrong and it trains badly
rather than failing loudly:

1. **Initialisation.** Zeros will not do here — every hidden unit would compute
   exactly the same thing forever, and stay identical, because they all receive
   the same gradient. You need to break that symmetry with random values, but
   scaled: too large and the ReLUs saturate before training starts. Look up **He
   initialisation** (`sqrt(2 / fan_in)`) and why the `2` is there for ReLU
   specifically.
2. **Momentum.** Keep a running velocity per parameter, `v = mom*v - lr*grad`,
   then `param += v`. It smooths the noisy per-batch gradient and roughly halves
   the epochs you need. `mom = 0.9`.
3. **Learning-rate decay.** A step size big enough to get near the minimum fast
   is too big to settle into it. Multiply the rate by ~`0.9` each epoch.

Start with `lr = 0.2`, batch 128, 25 epochs. Print the test accuracy every 5
epochs so you can see it converge — and so you notice if it diverges.

In [ ]:
rng = np.random.default_rng(0)
HIDDEN = 256

# TODO: initialise W1, b1, W2, b2  (He init on the weights, zeros on the biases)
W1 = b1 = W2 = b2 = None

EPOCHS, BATCH, LR, MOM = 25, 128, 0.2, 0.9
t0 = time.perf_counter()


def forward(x):
    """Return z1, a1, z2 (the logits)."""
    # TODO
    pass


# TODO: the training loop - forward, backward, momentum update, lr decay

pred_v4 = None      # TODO: argmax of the logits on Xte

if pred_v4 is not None:
    acc = (pred_v4 == yte).mean()
    results["V4 neural net (1 hidden)"] = acc
    n_params = W1.size + b1.size + W2.size + b2.size
    print(f"V4 neural net        test accuracy {acc:.4f}   "
          f"({time.perf_counter() - t0:.1f}s, {n_params:,} parameters)")

## Results

In [ ]:
print(f"{'model':<28}{'test accuracy':>15}{'errors':>12}")
print("-" * 55)
for name, a in results.items():
    n = 2000 if "k-NN" in name else 10000
    print(f"{name:<28}{a:>15.4f}{f'{round((1 - a) * n)}/{n // 1000}k':>12}")

The **ordering** is the lesson, more than any single number. Write down, in your
own words, what each step bought and what it cost:

- **V1 → V2** — what did dropping the averaging fix, and what did it cost you at
  prediction time?
- **V2 → V3** — accuracy went *down*. Argue for why you would still ship V3.
- **V3 → V4** — one hidden layer. Count the errors before and after, not the
  percentage; the fraction of remaining mistakes that disappeared is the number
  that makes the case for depth.

## Where the mistakes are

Two diagnostics, once V4 works. The confusion matrix shows *which* digits get
mistaken for which — and the top confusions are usually pairs you would predict
from how people write. Then look at the actual failing images: most are genuinely
ambiguous, the kind you would hesitate on too.

In [ ]:
if pred_v4 is not None:
    cm = np.zeros((10, 10), dtype=int)
    np.add.at(cm, (yte, pred_v4), 1)          # rows = truth, cols = prediction

    print("confusion matrix   (rows = true, cols = predicted)")
    print("     " + "".join(f"{d:>5}" for d in range(10)))
    for d in range(10):
        cells = "".join(f"{cm[d, j]:>5}" if d != j else f"{'.':>5}" for j in range(10))
        print(f"  {d}  {cells}   acc {cm[d, d] / cm[d].sum():.3f}")

    off = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j]
    print("\nmost common confusions:")
    for count, i, j in sorted(off, reverse=True)[:6]:
        print(f"  true {i} -> predicted {j}   {count} times")

In [ ]:
if pred_v4 is not None:
    wrong = np.flatnonzero(pred_v4 != yte)
    print(f"{len(wrong)} of {len(yte)} test images misclassified\n")
    for i in wrong[:3]:
        print(f"true {yte[i]}, predicted {pred_v4[i]}")
        show(test_x[i])
        print()

## Where this stops

Here is a test you can run in your head. Take a fixed random permutation and
shuffle all 784 pixels of every image the same way — scrambling each digit beyond
human recognition. **All four models score exactly the same.**

That is the ceiling, and it is around 98%. Every model here treats the image as
784 unordered numbers; none of them knows that pixel 400 is next to pixel 401.
More layers and more epochs move this very little.

Getting past it means telling the model that pixels have *neighbours*. That is a
convolution, and it is the next problem.